### PEAS/ Agent Analysis and Setup

```
                     Hidden State
                  (form / momentum)
                          |
                          v
        ------------------------------------------------
        |                 |                 |         |
        v                 v                 v         v
   Team Strength      Rest Context      Relative        Outcome
 (HomeStrength,    (HomeRest_bin,     Differences     (HOME_TEAM_WINS)
  AwayStrength)     AwayRest_bin)   (WinPctDiff_bin,
                                      RestDiff_bin)
```

### Conversion of tabular data into sequential format

In [1]:
import pandas as pd
import numpy as np

games = pd.read_csv("games.csv")
ranking = pd.read_csv("ranking.csv")

games["GAME_DATE_EST"] = pd.to_datetime(games["GAME_DATE_EST"])
ranking["STANDINGSDATE"] = pd.to_datetime(ranking["STANDINGSDATE"])

In [2]:
ranking = ranking.sort_values("STANDINGSDATE")
games = games.sort_values("GAME_DATE_EST")

home_strength = pd.merge_asof(
    games,
    ranking,
    left_on="GAME_DATE_EST",
    right_on="STANDINGSDATE",
    left_by="HOME_TEAM_ID",
    right_by="TEAM_ID",
    direction="backward"
)

away_strength = pd.merge_asof(
    games,
    ranking,
    left_on="GAME_DATE_EST",
    right_on="STANDINGSDATE",
    left_by="VISITOR_TEAM_ID",
    right_by="TEAM_ID",
    direction="backward",
    suffixes=("", "_away")
)

games = home_strength.merge(
    away_strength[["GAME_ID", "W_PCT"]],
    on="GAME_ID",
    suffixes=("_home", "_away")
)

In [5]:
# Strength variables
games["HomeStrength"] = pd.qcut(games["W_PCT_home"], q=3, labels=["Low", "Medium", "High"])

games["AwayStrength"] = pd.qcut(games["W_PCT_away"], q=3, labels=["Low", "Medium", "High"])

games["WinPctDiff"] = games["W_PCT_home"] - games["W_PCT_away"]

games["WinPctDiff_bin"] = pd.cut(
    games["WinPctDiff"],
    bins=[-1, -0.05, 0.05, 1],
    labels=["Negative", "Neutral", "Positive"]
)

In [4]:
def compute_rest_days(df, team_col):
    df = df.sort_values(["TEAM_ID", "GAME_DATE_EST"])
    df["PrevGameDate"] = df.groupby("TEAM_ID")["GAME_DATE_EST"].shift(1)
    df["RestDays"] = (df["GAME_DATE_EST"] - df["PrevGameDate"]).dt.days
    return df
home_games = games[["GAME_ID", "GAME_DATE_EST", "HOME_TEAM_ID"]].rename(
    columns={"HOME_TEAM_ID": "TEAM_ID"}
)
away_games = games[["GAME_ID", "GAME_DATE_EST", "VISITOR_TEAM_ID"]].rename(
    columns={"VISITOR_TEAM_ID": "TEAM_ID"}
)

home_games = compute_rest_days(home_games, "TEAM_ID")
away_games = compute_rest_days(away_games, "TEAM_ID")

games = games.merge(
    home_games[["GAME_ID", "RestDays"]],
    on="GAME_ID",
    how="left"
).rename(columns={"RestDays": "HomeRest"})

games = games.merge(
    away_games[["GAME_ID", "RestDays"]],
    on="GAME_ID",
    how="left"
).rename(columns={"RestDays": "AwayRest"})

In [6]:
# use computed rest days to discretize rest variables
rest_bins = [0, 1, 3, 10]
rest_labels = ["Short", "Normal", "Long"]

games["HomeRest_bin"] = pd.cut(games["HomeRest"], rest_bins, labels=rest_labels)
games["AwayRest_bin"] = pd.cut(games["AwayRest"], rest_bins, labels=rest_labels)

games["RestDiff"] = games["HomeRest"] - games["AwayRest"]

games["RestDiff_bin"] = pd.cut(
    games["RestDiff"],
    bins=[-10, -2, 2, 10],
    labels=["Negative", "Neutral", "Positive"]
)

In [7]:
# each sequence is a team's games ordered in time
def build_team_sequences(df):
    sequences = {}

    for team_id in pd.unique(
        df[["HOME_TEAM_ID", "VISITOR_TEAM_ID"]].values.ravel()
    ):
        team_games = df[
            (df["HOME_TEAM_ID"] == team_id) |
            (df["VISITOR_TEAM_ID"] == team_id)
        ].sort_values("GAME_DATE_EST")

        obs = team_games[
            [
                "HomeStrength",
                "AwayStrength",
                "HomeRest_bin",
                "AwayRest_bin",
                "WinPctDiff_bin",
                "RestDiff_bin",
                "HOME_TEAM_WINS"
            ]
        ].dropna()

        if len(obs) > 1:
            sequences[team_id] = obs.values.tolist()

    return sequences

In [8]:
team_sequences = build_team_sequences(games)

In [11]:
# first 3 games for first team
team_id, sequence = next(iter(team_sequences.items()))
print("Team ID:", team_id)
print("First 3 games:", sequence[:3])

Team ID: 1610612762
First 3 observations: [['Low', 'Low', 'Long', 'Short', 'Neutral', 'Positive', 0], ['Low', 'Low', 'Normal', 'Long', 'Neutral', 'Negative', 0], ['Low', 'Low', 'Normal', 'Long', 'Neutral', 'Negative', 0]]


## HMM Implementation with EM
Z_t ∈ {LowForm, MediumForm, HighForm}

composite observation variable: (HomeStrength, AwayStrength, HomeRest_bin, AwayRest_bin, WinPctDiff_bin, RestDiff_bin, HomeWin)

In [12]:
def encode_observations(df, obs_cols):
    obs_tuples = df[obs_cols].astype(str).agg("_".join, axis=1)
    symbols, encoded = np.unique(obs_tuples, return_inverse=True)
    return encoded, symbols

In [20]:
obs_cols = [
    "HomeStrength", "AwayStrength",
    "HomeRest_bin", "AwayRest_bin",
    "WinPctDiff_bin", "RestDiff_bin",
    "HOME_TEAM_WINS"
]

encoded_obs, obs_symbols = encode_observations(games, obs_cols)

In [16]:
def initialize_hmm(K, M, seed=0):
    rng = np.random.default_rng(seed)

    pi = rng.random(K)
    pi /= pi.sum()

    A = rng.random((K, K))
    A /= A.sum(axis=1, keepdims=True)

    B = rng.random((K, M))
    B /= B.sum(axis=1, keepdims=True)

    return pi, A, B

In [17]:
# E step
def forward_backward(obs, pi, A, B):
    T = len(obs)
    K = len(pi)

    alpha = np.zeros((T, K))
    beta = np.zeros((T, K))

    alpha[0] = pi * B[:, obs[0]]
    alpha[0] /= alpha[0].sum()

    for t in range(1, T):
        for j in range(K):
            alpha[t, j] = B[j, obs[t]] * np.sum(alpha[t-1] * A[:, j])
        alpha[t] /= alpha[t].sum()

    beta[-1] = 1
    for t in reversed(range(T - 1)):
        for i in range(K):
            beta[t, i] = np.sum(A[i] * B[:, obs[t+1]] * beta[t+1])
        beta[t] /= beta[t].sum()

    gamma = alpha * beta
    gamma /= gamma.sum(axis=1, keepdims=True)

    xi = np.zeros((T - 1, K, K))
    for t in range(T - 1):
        denom = np.sum(alpha[t][:, None] * A * B[:, obs[t+1]] * beta[t+1])
        for i in range(K):
            xi[t, i] = alpha[t, i] * A[i] * B[:, obs[t+1]] * beta[t+1] / denom

    return gamma, xi

In [27]:
# M step
def m_step(obs, gamma, xi, K, M):
    pi = gamma[0]

    A = xi.sum(axis=0)
    A /= A.sum(axis=1, keepdims=True)

    B = np.zeros((K, M))
    for k in range(K):
        for t, o in enumerate(obs):
            B[k, o] += gamma[t, k]
        B[k] /= B[k].sum()

    return pi, A, B

In [19]:
# EM training
def train_hmm(obs, K, M, n_iters=20):
    pi, A, B = initialize_hmm(K, M)

    for _ in range(n_iters):
        gamma, xi = forward_backward(obs, pi, A, B)
        pi, A, B = m_step(obs, gamma, xi, K, M)

    return pi, A, B

In [28]:
# Train
team_id, seq = next(iter(team_sequences.items()))

seq_df = pd.DataFrame(seq, columns=obs_cols)
obs_encoded, obs_symbols = encode_observations(seq_df, obs_cols)

K = 3
M = len(obs_symbols)

pi_hat, A_hat, B_hat = train_hmm(obs_encoded, K, M)

In [29]:
print("π:")
print(pi_hat)

print("\nA:")
print(A_hat)

print("\nB:")
print(B_hat)

π:
[0. 1. 0.]

A:
[[4.42274312e-05 2.41945736e-01 7.58010036e-01]
 [3.69727462e-01 3.22129774e-01 3.08142764e-01]
 [7.30115587e-01 2.69883182e-01 1.23074506e-06]]

B:
[[9.16794599e-030 2.20148286e-003 3.32743445e-003 3.13371333e-009
  5.34743741e-003 8.82485836e-003 2.61989108e-002 7.87647657e-051
  3.59535325e-003 2.19153630e-024 7.59025463e-008 1.79863221e-003
  1.64363346e-005 2.74272504e-003 9.93414508e-005 1.79853003e-003
  2.60747109e-007 6.76440551e-005 5.76325796e-003 9.01168638e-013
  5.39762128e-003 5.73361347e-019 6.02812552e-027 1.79920625e-003
  2.24053534e-008 6.73401735e-042 1.22614211e-006 1.24007705e-018
  5.37901872e-003 1.71016204e-003 1.72874860e-003 1.79920714e-003
  1.43493464e-002 5.65877201e-003 3.01711761e-003 1.10374727e-003
  5.36226514e-004 1.92387390e-015 8.82098328e-003 3.56248197e-036
  2.91318572e-004 6.05790518e-006 1.17535763e-050 1.81023899e-003
  3.22905640e-016 1.11511204e-075 1.79920712e-003 3.59819561e-003
  6.31370901e-004 1.00241193e-003 4.05113

## Inference

In [39]:
def forward_alg(observations, pi, A, B):
    T = len(observations)
    N = len(pi)

    alpha = np.zeros((T, N))
    alpha[0] = pi * B[:, observations[0]]
    alpha[0] /= alpha[0].sum()

    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = B[j, observations[t]] * np.sum(
                alpha[t-1] * A[:, j]
            )
        alpha[t] /= alpha[t].sum()

    return alpha

In [45]:
def predict_home_win_prob(alpha_prev, A, B, home_win_index):
    state_pred = alpha_prev @ A

    p_home_win = state_pred @ B[:, home_win_indices].sum(axis=1)

    return float(p_home_win)

In [46]:
def hmm_game_predictions(observations, pi, A, B, home_win_index):

    alpha = forward_alg(observations, pi, A, B)
    probs = []
    preds = []
    for t in range(1, len(observations)):
        p = predict_home_win_prob(alpha[t-1], A, B, home_win_index)
        probs.append(p)
        preds.append(int(p >= 0.5))

    return np.array(probs), np.array(preds)

In [48]:
team_id, seq = next(iter(team_sequences.items()))
seq_df = pd.DataFrame(seq, columns=obs_cols)
obs_encoded, obs_symbols = encode_observations(seq_df, obs_cols)

# Identify which encoded symbols correspond to HOME_TEAM_WINS == 1
home_win_mask    = np.array([s.endswith('_1') for s in obs_symbols])
home_win_indices = np.where(home_win_mask)[0]

print(f"Total observation symbols : {len(obs_symbols)}")
print(f"Symbols encoding home win : {home_win_indices.sum()}")

Total observation symbols : 284
Symbols encoding home win : 20757


In [49]:
probs, preds = hmm_game_predictions(obs_encoded, pi_hat, A_hat, B_hat,
                                    home_win_indices)

true_labels = seq_df['HOME_TEAM_WINS'].astype(int).values
y_true  = true_labels[1:]

accuracy = np.mean(preds == y_true)
print(f"Team {team_id}")
print(f"  Games evaluated          : {len(y_true)}")
print(f"  Forward-filter accuracy  : {accuracy:.3f}")
print()
print("  Predicted win probs (first 10):", np.round(probs[:10], 3))
print("  Predictions          (first 10):", preds[:10])
print("  Ground truth         (first 10):", y_true[:10])


Team 1610612762
  Games evaluated          : 1523
  Forward-filter accuracy  : 0.627

  Predicted win probs (first 10): [0.631 0.628 0.628 0.628 0.622 0.628 0.623 0.631 0.628 0.622]
  Predictions          (first 10): [1 1 1 1 1 1 1 1 1 1]
  Ground truth         (first 10): [0 0 0 0 1 1 1 1 1 1]


In [50]:
# Predictions for all teams
all_true, all_pred = [], []

for tid, seq in team_sequences.items():
    if len(seq) < 3:
        continue
    s_df   = pd.DataFrame(seq, columns=obs_cols)
    enc, _ = encode_observations(s_df, obs_cols)

    # Identify home-win indices for this team's local symbol set
    syms = _
    hw_idx = np.where(np.array([s.endswith('_1') for s in syms]))[0]
    if len(hw_idx) == 0:
        continue

    try:
        pi_t, A_t, B_t = train_hmm(enc, K=3, M=len(syms), n_iters=20)
        pr, pd_ = hmm_game_predictions(enc, pi_t, A_t, B_t, hw_idx)
        yt = s_df['HOME_TEAM_WINS'].astype(int).values[1:]
        all_true.extend(yt.tolist())
        all_pred.extend(pd_.tolist())
    except Exception:
        continue

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"Overall across {len(team_sequences)} teams")
print(f"  Total predictions : {len(all_true)}")
print(f"  Accuracy          : {np.mean(all_true == all_pred):.3f}")
print(f"  Baseline (always predict home win): {all_true.mean():.3f}")


Overall across 30 teams
  Total predictions : 41077
  Accuracy          : 0.511
  Baseline (always predict home win): 0.595


## Evaluation

In [51]:
def forward_loglik(observations, pi, A, B):
    T = len(observations)
    K = len(pi)
    alpha = np.zeros(K)
    log_lik = 0.0

    alpha = pi * B[:, observations[0]]
    c = alpha.sum()
    if c < 1e-300:
        return -np.inf
    alpha /= c
    log_lik += np.log(c)

    for t in range(1, T):
        alpha_new = np.array([
            B[j, observations[t]] * np.sum(alpha * A[:, j])
            for j in range(K)
        ])
        c = alpha_new.sum()
        if c < 1e-300:
            return -np.inf
        alpha = alpha_new / c
        log_lik += np.log(c)

    return log_lik


pi_init, A_init, B_init = initialize_hmm(K=3, M=len(obs_symbols), seed=0)
ll_before = forward_loglik(obs_encoded, pi_init, A_init, B_init)
ll_after  = forward_loglik(obs_encoded, pi_hat,  A_hat,  B_hat)

print(f"Log-likelihood before EM : {ll_before:.3f}")
print(f"Log-likelihood after  EM : {ll_after:.3f}")
print(f"Improvement              : {ll_after - ll_before:+.3f}  (positive = EM improved fit)")
print(f"Per-observation avg      : {ll_after / len(obs_encoded):.4f}  nats/game")


Log-likelihood before EM : -8716.371
Log-likelihood after  EM : -7590.372
Improvement              : +1125.999  (positive = EM improved fit)
Per-observation avg      : -4.9806  nats/game
